# NOTEBOOK 04 – TRÚC CHI | CHƯƠNG 4: MÔ HÌNH & THỰC NGHIỆM

**Phụ trách:** 4.1–4.5 · Issue **#5** · Hạn báo cáo/Validation: 22/11/2026; bàn giao dự đoán Test cho Vĩnh Chi trước 24/11.

**CHỈ HUẤN LUYỆN:** Dummy Classifier. **Tiếp nhận RF** từ Ngọc Lan và **MLP** từ Tấn Lên (TV1), không fit lại cả RF/MLP.

**Đầu vào** từ Lam Linh: `data/processed/orders_model_ready.csv`. **Bàn giao từ Tấn Lên:** `models/tan_len_mlp.joblib`, `mlp_val_predictions.csv`, `mlp_metadata.json`.

**Bàn giao từ Ngọc Lan:** `models/ngoc_lan_random_forest.joblib`, `data/processed/rf_val_predictions.csv`, `rf_metadata.json` dùng cùng split.

**Đầu ra**: `test_predictions.csv`, `experiment_metadata.json`, các model `joblib` (không push), bảng tham số và ngưỡng trên Validation.

[Lý thuyết Ch4](../docs/theory/04_truc_chi_ch4.md) · [Giao thức thực nghiệm](../docs/EXPERIMENT_PROTOCOL.md).

> Chạy **chỉ sau khi Ch3 bàn giao** và audit. Không đưa thông tin tương lai vào X. Không báo kết quả trước khi chạy; các cell ở đây là code gợi ý chuẩn hóa để hoàn thiện.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 4.1 Chia Train/Validation/Test thống nhất (Trúc Chi phụ trách)

Trúc Chi tạo split duy nhất từ `src/experiment.py` và bàn giao `split_id` cho **Ngọc Lan (RF)**, **Tấn Lên (MLP)**.

Thứ tự thực hành: Lam Linh dữ liệu → Trúc Chi split → Ngọc Lan RF và Tấn Lên MLP trên cùng Train/Validation → Trúc Chi chỉ fit Dummy, tích hợp RF/MLP → Vĩnh Chi kiểm chứng Test độc lập.

**Không huấn luyện lại RF hoặc MLP trong notebook 04; không thay đổi feature whitelist, nhãn, cutoff.**

In [ ]:
from src.experiment import (get_shared_splits, NUMERIC_FEATURES, CATEGORY_FEATURES,
                            SAFE_FEATURES, choose_threshold_on_validation)
splits = get_shared_splits(PROCESSED)
train, val, test = splits["train"], splits["val"], splits["test"]
split_info = splits["metadata"]
split_id = split_info["split_id"]
FEATURES = list(SAFE_FEATURES)
print("Shared split id:", split_id)
display(pd.DataFrame([
    {"split":label, "n":len(splits[label]),
     "late_rate":splits[label]["is_late"].mean()}
    for label in ("train","val","test")
]))
print("Ngọc Lan (RF), Tấn Lên (MLP), Trúc Chi (Dummy) dùng cùng split_id.")

## 4.2 Tiền xử lý và huấn luyện Dummy Classifier – Trúc Chi (TV4)

Trúc Chi chỉ fit **Dummy Classifier**, tạo baseline trên Train chung và tính xác suất Validation. Preprocessing phải fit Train-only.

RF do Ngọc Lan đã train; MLP do Tấn Lên đã train. Trúc Chi tiếp nhận các artifact, kiểm tra metadata và tích hợp thay vì train thêm.

In [ ]:
import json
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score

FEATURES = list(SAFE_FEATURES)
X_train, y_train = train[FEATURES], train["is_late"].astype(int)
X_val, y_val = val[FEATURES], val["is_late"].astype(int)
X_test, y_test = test[FEATURES], test["is_late"].astype(int)
assert y_train.nunique()==2 and y_val.nunique()==2
dummy_prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), CATEGORY_FEATURES),
])
dummy = Pipeline([
    ("preprocess", dummy_prep),
    ("dummy", DummyClassifier(strategy="prior")),
])
dummy.fit(X_train, y_train)  # Trúc Chi CHỈ fit Dummy
p_dummy_val = dummy.predict_proba(X_val)[:,1]
print("TV4 đã huấn luyện Dummy; RF và MLP nhận artifact từ Ngọc Lan và Tấn Lên.")

## 4.3 Tiếp nhận Random Forest của Ngọc Lan; 4.4 tiếp nhận MLP của Tấn Lên

**4.3 Random Forest:** Ngọc Lan huấn luyện tại notebook 02, Trúc Chi nhận `ngoc_lan_random_forest.joblib`, `rf_val_predictions.csv`, `rf_metadata.json`.

**4.4 MLP:** Tấn Lên trực tiếp huấn luyện tại notebook 01 và bàn giao `tan_len_mlp.joblib`, `mlp_val_predictions.csv`, `mlp_metadata.json`. Trúc Chi vẫn là người phụ trách biên soạn mục 4.4 trong **Chương 4**.

Kiểm tra bắt buộc: `split_id`, dataset SHA256, feature whitelist, Validation order_id/y_true, xác suất `predict_proba` phải trùng file bàn giao. **Không được gọi `fit` với RF hay MLP.**

In [ ]:
rf_path = ROOT / "models" / "ngoc_lan_random_forest.joblib"
mlp_path = ROOT / "models" / "tan_len_mlp.joblib"
rf_meta_path = PROCESSED / "rf_metadata.json"
mlp_meta_path = PROCESSED / "mlp_metadata.json"
rf_val_path = PROCESSED / "rf_val_predictions.csv"
mlp_val_path = PROCESSED / "mlp_val_predictions.csv"
required = (rf_path, mlp_path, rf_meta_path, mlp_meta_path, rf_val_path, mlp_val_path)
missing = [p.name for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Chưa nhận RF/MLP từ Ngọc Lan/Tấn Lên: " + ", ".join(missing))
rf_meta = json.loads(rf_meta_path.read_text(encoding="utf-8"))
mlp_meta = json.loads(mlp_meta_path.read_text(encoding="utf-8"))
rf_val_file = pd.read_csv(rf_val_path)
mlp_val_file = pd.read_csv(mlp_val_path)
for name, meta, owner, validation_file, prob_col in [
    ("rf", rf_meta, "Ngọc Lan", rf_val_file, "p_rf"),
    ("mlp", mlp_meta, "Tấn Lên", mlp_val_file, "p_mlp"),
]:
    assert meta["model"] == name and meta["owner"] == owner
    assert meta["split_id"] == split_id
    assert meta["dataset_sha256"] == split_info["csv_sha256"]
    assert meta["features"] == FEATURES
    assert meta["validation_n"] == len(val)
    assert np.array_equal(validation_file["order_id"].to_numpy(), val["order_id"].to_numpy())
    assert np.array_equal(validation_file["y_true"].to_numpy(), y_val.to_numpy())
rf = joblib.load(rf_path)
mlp = joblib.load(mlp_path)
p_rf = rf.predict_proba(X_val)[:,1]
p_mlp = mlp.predict_proba(X_val)[:,1]
assert np.allclose(p_rf, rf_val_file["p_rf"].to_numpy(), atol=1e-7)
assert np.allclose(p_mlp, mlp_val_file["p_mlp"].to_numpy(), atol=1e-7)
models = {"dummy": dummy, "rf": rf, "mlp": mlp}
validation_prob = {"dummy": p_dummy_val, "rf": p_rf, "mlp": p_mlp}
validation_results = [
    {"model": name,
     "owner": {"rf":"Ngọc Lan", "mlp":"Tấn Lên", "dummy":"Trúc Chi"}[name],
     "val_AP": float(average_precision_score(y_val, validation_prob[name]))}
    for name in ("dummy", "rf", "mlp")
]
display(pd.DataFrame(validation_results).sort_values("val_AP",ascending=False))
print("TV4 chỉ fit Dummy, RF/MLP đã được kiểm chứng mà không fit lại.")

## 4.5 Tổng hợp và chọn champion/threshold chỉ trên Validation

- Random Forest: lấy threshold do Ngọc Lan chốt từ Validation.
- MLP: lấy threshold do Tấn Lên chốt từ Validation.
- Dummy: Trúc Chi chọn threshold theo F1 Validation.
- Trúc Chi so sánh AP Validation của RF/MLP để chọn champion. Không dùng Test quyết định bất kỳ tham số nào.

In [ ]:
from src.experiment import classification_metrics
thresholds = {
    "rf": float(rf_meta["validation_metrics"]["threshold"]),
    "mlp": float(mlp_meta["validation_metrics"]["threshold"]),
}
dummy_val_metrics = choose_threshold_on_validation(y_val, validation_prob["dummy"])
thresholds["dummy"] = float(dummy_val_metrics["threshold"])
val_metric_rows = []
owners = {"rf":"Ngọc Lan","mlp":"Tấn Lên","dummy":"Trúc Chi"}
for name in ("dummy","rf","mlp"):
    result = classification_metrics(y_val, validation_prob[name], thresholds[name])
    val_metric_rows.append({"model":name, "owner":owners[name],
                            **result, "dataset":"Validation"})
for name, meta in (("rf",rf_meta),("mlp",mlp_meta)):
    actual_ap = next(row["AP"] for row in val_metric_rows if row["model"]==name)
    assert np.isclose(actual_ap, float(meta["validation_metrics"]["AP"]), atol=1e-8)
display(pd.DataFrame(val_metric_rows))
champion = max((r for r in validation_results if r["model"] in ("rf","mlp")),
               key=lambda r:r["val_AP"])["model"]
chosen_threshold = thresholds[champion]
print("Champion dựa trên AP Validation:", champion, "threshold:", chosen_threshold)

## 4.6 Đóng băng dự đoán Test và bàn giao Vĩnh Chi

Chỉ sau khi nhận RF của Ngọc Lan, MLP của Tấn Lên và huấn luyện Dummy, Trúc Chi mới xuất Test predictions.

Lưu `test_predictions.csv` (ba cột xác suất) + `experiment_metadata.json` ghi ba model owners và `split_id`, `models/dummy_truc_chi.joblib`, `models/champion.joblib`. **Không tạo file MLP mới; dùng artifact Tấn Lên đã bàn giao.**

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
test_probabilities = {
    name: models[name].predict_proba(X_test)[:,1]
    for name in ("dummy","rf","mlp")
}
predictions = pd.DataFrame({
    "order_id":test["order_id"].to_numpy(),
    "y_true":y_test.to_numpy(),
    "p_dummy":test_probabilities["dummy"],
    "p_rf":test_probabilities["rf"],
    "p_mlp":test_probabilities["mlp"],
})
predictions["y_pred_champion"] = (
    predictions["p_"+champion].to_numpy() >= chosen_threshold
).astype(int)
assert predictions["order_id"].is_unique and len(predictions)==len(test)
predictions.to_csv(PROCESSED / "test_predictions.csv", index=False)
joblib.dump(dummy, MODEL_DIR / "dummy_truc_chi.joblib")
joblib.dump(models[champion], MODEL_DIR / "champion.joblib")
meta = {
    "prediction_time":"order_approved_at", "features":FEATURES,
    "split_id":split_id, "split_metadata":split_info,
    "model_owners":{"rf":"Ngọc Lan","dummy":"Trúc Chi","mlp":"Tấn Lên"},
    "rf_source":"models/ngoc_lan_random_forest.joblib",
    "mlp_source":"models/tan_len_mlp.joblib",
    "champion":champion, "thresholds_from_val":thresholds,
    "chosen_threshold":chosen_threshold,
    "validation_AP":{r["model"]:r["val_AP"] for r in validation_results},
    "test_n":len(test), "random_state":42,
    "label":"DATE(actual)>DATE(estimated) on delivered labeled orders",
}
(PROCESSED / "experiment_metadata.json").write_text(
    json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8"
)
print("Trúc Chi đã tích hợp RF Ngọc Lan + MLP Tấn Lên + Dummy TV4; frozen Test cho Vĩnh Chi.")

## 4.7 Checklist nghiệm thu Chương 4
- [ ] Split Train/Validation/Test chung cho RF, MLP, Dummy có split_id, cutoff, dữ liệu SHA256.
- [ ] Trúc Chi CHỈ gọi fit Dummy, không gọi fit RF hoặc MLP.
- [ ] Đã nhận RF artifact/metadata/Validation của Ngọc Lan.
- [ ] Đã nhận MLP artifact/metadata/Validation của Tấn Lên (notebook 01).
- [ ] 4.4 trình bày kiến trúc, cấu hình, training loss/Validation của TV1, không nhận công train của TV4.
- [ ] Đã kiểm tra order_id, y_true, Validation p_rf và p_mlp trước khi tích hợp.
- [ ] Champion và thresholds chốt từ Validation; exported frozen Test, metadata và Dummy fitted artifact.
- [ ] Vĩnh Chi nhận Test và toàn bộ fitted model files, không tuning lại Test.